<a href="https://colab.research.google.com/github/sejal04-hash/PORTFOLIO/blob/master/Student_Test_Scores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv("/content/train.csv")

In [ ]:
df.isnull().sum()

,0
id,0
age,0
gender,0
course,0
study_hours,0
class_attendance,0
internet_access,0
sleep_hours,0
sleep_quality,0
study_method,0


In [ ]:
df.head()

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy,78.3
1,1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,moderate,46.7
2,2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,moderate,99.0
3,3,19,male,b.sc,2.00,49.5,yes,8.3,average,group study,high,moderate,63.9
4,4,23,male,bca,7.65,86.9,yes,9.6,good,self-study,high,easy,100.0


In [ ]:
df['exam_difficulty'] = df['exam_difficulty'].fillna(df['exam_difficulty'].mode()[0])

In [ ]:
df.isnull().sum()

,0
id,0
age,0
gender,0
course,0
study_hours,0
class_attendance,0
internet_access,0
sleep_hours,0
sleep_quality,0
study_method,0


In [ ]:
df['exam_score']= df['exam_score'].fillna(df['exam_score'].mean())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [ ]:
target = "exam_score"

y = df[target]
x = df.drop(columns = ["target","id"])

# categorical features

cat_cols = x.select_dtypes(include = "object").columns.tolist()

print("categorical columns:")
print(cat_cols)

NameError: name 'df' is not defined

In [ ]:
x_train,x_val , y_train,y_val = train_test_split(x, y , random_state = 1  )

In [ ]:
! pip install lightgbm xgboost catboost scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.0 MB/s eta 0:00:00


In [16]:

# =========================================================
# STUDENT TEST SCORE - ENSEMBLE LEARNING
# LightGBM + XGBoost + CatBoost + Ridge Stacking
# =========================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import Ridge

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor


# =========================================================
# 1. LOAD DATA
# =========================================================

TRAIN_PATH = "/train.csv"
TEST_PATH = "/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("=" * 70)
print("DATA INFORMATION")
print("=" * 70)

print("Train shape:", train.shape)
print("Test shape :", test.shape)

print("\nTrain columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())


# =========================================================
# 2. TARGET
# =========================================================

TARGET = "exam_score"


# =========================================================
# 3. CLEAN TARGET
# =========================================================

print("\n" + "=" * 70)
print("TARGET CHECK")
print("=" * 70)

# Convert target to numeric
train[TARGET] = pd.to_numeric(
    train[TARGET],
    errors="coerce"
)

# Replace infinity with NaN
train[TARGET] = train[TARGET].replace(
    [np.inf, -np.inf],
    np.nan
)

print(
    "NaN target values:",
    train[TARGET].isna().sum()
)

print(
    "Infinite target values:",
    np.isinf(train[TARGET]).sum()
)


# ---------------------------------------------------------
# Remove rows with invalid target
# ---------------------------------------------------------

before_rows = len(train)

train = train.dropna(
    subset=[TARGET]
).reset_index(drop=True)

after_rows = len(train)

print(
    "Removed invalid target rows:",
    before_rows - after_rows
)


# Final target validation
assert train[TARGET].notna().all()
assert np.isfinite(train[TARGET]).all()

print("Target validation: PASSED")


# =========================================================
# 4. CREATE FEATURES AND TARGET
# =========================================================

y = train[TARGET].astype(np.float32)

X = train.drop(
    columns=[TARGET, "id"]
).copy()

X_test = test.drop(
    columns=["id"]
).copy()


print("\nX shape     :", X.shape)
print("X_test shape:", X_test.shape)


# =========================================================
# 5. IDENTIFY COLUMN TYPES
# =========================================================

cat_cols = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

num_cols = [
    col for col in X.columns
    if col not in cat_cols
]


print("\nCategorical columns:")
print(cat_cols)

print("\nNumerical columns:")
print(num_cols)


# =========================================================
# 6. HANDLE MISSING VALUES
# =========================================================

print("\n" + "=" * 70)
print("MISSING VALUE HANDLING")
print("=" * 70)


# ---------------------------------------------------------
# 6.1 CATEGORICAL FEATURES
# ---------------------------------------------------------
# CatBoost does NOT accept NaN as a categorical value.
# Therefore replace missing categorical values with "Missing".
# ---------------------------------------------------------

for col in cat_cols:

    X[col] = (
        X[col]
        .astype("string")
        .fillna("Missing")
        .astype(str)
    )

    X_test[col] = (
        X_test[col]
        .astype("string")
        .fillna("Missing")
        .astype(str)
    )


# ---------------------------------------------------------
# 6.2 NUMERICAL FEATURES
# ---------------------------------------------------------
# Use TRAINING median.
# The same training median is applied to test data.
# ---------------------------------------------------------

for col in num_cols:

    X[col] = pd.to_numeric(
        X[col],
        errors="coerce"
    )

    X_test[col] = pd.to_numeric(
        X_test[col],
        errors="coerce"
    )

    median_value = X[col].median()

    X[col] = X[col].fillna(
        median_value
    )

    X_test[col] = X_test[col].fillna(
        median_value
    )


# =========================================================
# 7. FINAL FEATURE VALIDATION
# =========================================================

print("\nMissing values in X:")
print(X.isna().sum())

print("\nMissing values in X_test:")
print(X_test.isna().sum())

total_train_missing = X.isna().sum().sum()
total_test_missing = X_test.isna().sum().sum()

print(
    "\nTotal missing values in X:",
    total_train_missing
)

print(
    "Total missing values in X_test:",
    total_test_missing
)

assert total_train_missing == 0
assert total_test_missing == 0

print("\nFeature missing-value validation: PASSED")


# =========================================================
# 8. ORDINAL ENCODING FOR XGBOOST
# =========================================================

print("\n" + "=" * 70)
print("ENCODING CATEGORICAL FEATURES")
print("=" * 70)


encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)


X_encoded = X.copy()
X_test_encoded = X_test.copy()


if len(cat_cols) > 0:

    X_encoded[cat_cols] = encoder.fit_transform(
        X[cat_cols]
    )

    X_test_encoded[cat_cols] = encoder.transform(
        X_test[cat_cols]
    )


# Convert all features to float32
X_encoded = X_encoded.astype(
    np.float32
)

X_test_encoded = X_test_encoded.astype(
    np.float32
)


print("Encoding completed.")


# =========================================================
# 9. CROSS VALIDATION
# =========================================================

N_SPLITS = 5

kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=42
)


# =========================================================
# 10. OOF ARRAYS
# =========================================================

oof_lgb = np.zeros(
    len(X),
    dtype=np.float32
)

oof_xgb = np.zeros(
    len(X),
    dtype=np.float32
)

oof_cat = np.zeros(
    len(X),
    dtype=np.float32
)


test_lgb = np.zeros(
    len(X_test),
    dtype=np.float32
)

test_xgb = np.zeros(
    len(X_test),
    dtype=np.float32
)

test_cat = np.zeros(
    len(X_test),
    dtype=np.float32
)


# =========================================================
# 11. CATBOOST CATEGORICAL INDICES
# =========================================================

cat_indices = [
    X.columns.get_loc(col)
    for col in cat_cols
]


# =========================================================
# 12. TRAIN 5-FOLD ENSEMBLE
# =========================================================

for fold, (train_idx, val_idx) in enumerate(
    kf.split(X),
    start=1
):

    print("\n")
    print("=" * 70)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("=" * 70)


    # =====================================================
    # SPLIT DATA
    # =====================================================

    X_train = X.iloc[
        train_idx
    ].copy()

    X_val = X.iloc[
        val_idx
    ].copy()


    X_train_enc = X_encoded.iloc[
        train_idx
    ]

    X_val_enc = X_encoded.iloc[
        val_idx
    ]


    y_train = y.iloc[
        train_idx
    ]

    y_val = y.iloc[
        val_idx
    ]


    # =====================================================
    # 12.1 LIGHTGBM
    # =====================================================

    print("\nTraining LightGBM...")

    lgb_model = LGBMRegressor(

        n_estimators=1500,

        learning_rate=0.03,

        num_leaves=63,

        max_depth=-1,

        subsample=0.85,

        colsample_bytree=0.90,

        reg_alpha=0.1,

        reg_lambda=1.0,

        objective="regression",

        random_state=42 + fold,

        n_jobs=-1,

        verbosity=-1
    )


    # -----------------------------------------------------
    # Prepare categorical columns for LightGBM
    # -----------------------------------------------------

    X_train_lgb = X_train.copy()
    X_val_lgb = X_val.copy()
    X_test_lgb = X_test.copy()


    for col in cat_cols:

        # Make training column categorical
        X_train_lgb[col] = (
            X_train_lgb[col]
            .astype("category")
        )

        # Get training categories
        categories = (
            X_train_lgb[col]
            .cat.categories
        )

        # Apply same categories to validation
        X_val_lgb[col] = (
            X_val_lgb[col]
            .astype("category")
            .cat.set_categories(
                categories
            )
        )

        # Apply same categories to test
        X_test_lgb[col] = (
            X_test_lgb[col]
            .astype("category")
            .cat.set_categories(
                categories
            )
        )


    # -----------------------------------------------------
    # Train LightGBM
    # -----------------------------------------------------

    lgb_model.fit(

        X_train_lgb,

        y_train,

        categorical_feature=cat_cols,

        eval_set=[
            (
                X_val_lgb,
                y_val
            )
        ]
    )


    # -----------------------------------------------------
    # Validation prediction
    # -----------------------------------------------------

    lgb_val_pred = lgb_model.predict(
        X_val_lgb
    )

    oof_lgb[val_idx] = (
        lgb_val_pred
    )


    # -----------------------------------------------------
    # Test prediction
    # -----------------------------------------------------

    lgb_test_pred = lgb_model.predict(
        X_test_lgb
    )

    test_lgb += (
        lgb_test_pred / N_SPLITS
    )


    # =====================================================
    # 12.2 XGBOOST
    # =====================================================

    print("Training XGBoost...")


    xgb_model = XGBRegressor(

        n_estimators=1500,

        max_depth=8,

        learning_rate=0.04,

        subsample=0.85,

        colsample_bytree=0.90,

        min_child_weight=5,

        reg_alpha=0.1,

        reg_lambda=2.0,

        objective="reg:squarederror",

        eval_metric="rmse",

        tree_method="hist",

        random_state=42 + fold,

        n_jobs=-1
    )


    # -----------------------------------------------------
    # Train XGBoost
    # -----------------------------------------------------

    xgb_model.fit(

        X_train_enc,

        y_train,

        eval_set=[
            (
                X_val_enc,
                y_val
            )
        ],

        verbose=False
    )


    # -----------------------------------------------------
    # Validation prediction
    # -----------------------------------------------------

    xgb_val_pred = xgb_model.predict(
        X_val_enc
    )

    oof_xgb[val_idx] = (
        xgb_val_pred
    )


    # -----------------------------------------------------
    # Test prediction
    # -----------------------------------------------------

    xgb_test_pred = xgb_model.predict(
        X_test_encoded
    )

    test_xgb += (
        xgb_test_pred / N_SPLITS
    )


    # =====================================================
    # 12.3 CATBOOST
    # =====================================================

    print("Training CatBoost...")


    cat_model = CatBoostRegressor(

        iterations=1200,

        depth=7,

        learning_rate=0.05,

        loss_function="RMSE",

        eval_metric="RMSE",

        l2_leaf_reg=5,

        random_seed=42 + fold,

        thread_count=-1,

        verbose=False,

        allow_writing_files=False
    )


    # -----------------------------------------------------
    # Train CatBoost
    # -----------------------------------------------------

    cat_model.fit(

        X_train,

        y_train,

        cat_features=cat_indices,

        eval_set=(
            X_val,
            y_val
        ),

        early_stopping_rounds=80,

        verbose=False
    )


    # -----------------------------------------------------
    # Validation prediction
    # -----------------------------------------------------

    cat_val_pred = cat_model.predict(
        X_val
    )

    oof_cat[val_idx] = (
        cat_val_pred
    )


    # -----------------------------------------------------
    # Test prediction
    # -----------------------------------------------------

    cat_test_pred = cat_model.predict(
        X_test
    )

    test_cat += (
        cat_test_pred / N_SPLITS
    )


    # =====================================================
    # FOLD RMSE
    # =====================================================

    lgb_rmse = np.sqrt(
        mean_squared_error(
            y_val,
            lgb_val_pred
        )
    )

    xgb_rmse = np.sqrt(
        mean_squared_error(
            y_val,
            xgb_val_pred
        )
    )

    cat_rmse = np.sqrt(
        mean_squared_error(
            y_val,
            cat_val_pred
        )
    )


    print("\n")
    print("-" * 50)
    print(f"Fold {fold} Results")
    print("-" * 50)

    print(
        f"LightGBM : {lgb_rmse:.6f}"
    )

    print(
        f"XGBoost  : {xgb_rmse:.6f}"
    )

    print(
        f"CatBoost : {cat_rmse:.6f}"
    )


# =========================================================
# 13. OVERALL OOF SCORES
# =========================================================

print("\n")
print("=" * 70)
print("OVERALL OOF RESULTS")
print("=" * 70)


lgb_oof_rmse = np.sqrt(
    mean_squared_error(
        y,
        oof_lgb
    )
)

xgb_oof_rmse = np.sqrt(
    mean_squared_error(
        y,
        oof_xgb
    )
)

cat_oof_rmse = np.sqrt(
    mean_squared_error(
        y,
        oof_cat
    )
)


print(
    f"LightGBM OOF RMSE : {lgb_oof_rmse:.6f}"
)

print(
    f"XGBoost OOF RMSE  : {xgb_oof_rmse:.6f}"
)

print(
    f"CatBoost OOF RMSE : {cat_oof_rmse:.6f}"
)


# =========================================================
# 14. SIMPLE WEIGHTED ENSEMBLE
# =========================================================

print("\n")
print("=" * 70)
print("WEIGHTED ENSEMBLE")
print("=" * 70)


# Initial weights
LGB_WEIGHT = 0.40
XGB_WEIGHT = 0.35
CAT_WEIGHT = 0.25


blend_oof = (

    LGB_WEIGHT * oof_lgb +

    XGB_WEIGHT * oof_xgb +

    CAT_WEIGHT * oof_cat
)


blend_rmse = np.sqrt(
    mean_squared_error(
        y,
        blend_oof
    )
)


print(
    f"LightGBM weight : {LGB_WEIGHT}"
)

print(
    f"XGBoost weight  : {XGB_WEIGHT}"
)

print(
    f"CatBoost weight : {CAT_WEIGHT}"
)

print(
    f"\nWeighted Ensemble RMSE: {blend_rmse:.6f}"
)


# =========================================================
# 15. RIDGE STACKING
# =========================================================

print("\n")
print("=" * 70)
print("RIDGE STACKING")
print("=" * 70)


# ---------------------------------------------------------
# OOF predictions become features for meta-model
# ---------------------------------------------------------

stack_X = np.column_stack([

    oof_lgb,

    oof_xgb,

    oof_cat

])


# Test predictions become meta-test features

stack_test = np.column_stack([

    test_lgb,

    test_xgb,

    test_cat

])


print(
    "Stack training shape:",
    stack_X.shape
)

print(
    "Stack test shape:",
    stack_test.shape
)


# =========================================================
# 16. RIDGE META MODEL
# =========================================================

meta_model = Ridge(
    alpha=10.0
)


meta_model.fit(
    stack_X,
    y
)


# =========================================================
# 17. STACKING OOF PREDICTION
# =========================================================

stack_oof_pred = meta_model.predict(
    stack_X
)


stack_rmse = np.sqrt(
    mean_squared_error(
        y,
        stack_oof_pred
    )
)


print(
    f"\nRidge Stacking RMSE: {stack_rmse:.6f}"
)


# =========================================================
# 18. RIDGE COEFFICIENTS
# =========================================================

print("\nRidge coefficients:")

print(
    "LightGBM:",
    meta_model.coef_[0]
)

print(
    "XGBoost :",
    meta_model.coef_[1]
)

print(
    "CatBoost:",
    meta_model.coef_[2]
)

print(
    "Intercept:",
    meta_model.intercept_
)


# =========================================================
# 19. FINAL TEST PREDICTION
# =========================================================

print("\n")
print("=" * 70)
print("FINAL TEST PREDICTION")
print("=" * 70)


final_pred = meta_model.predict(
    stack_test
)


# Make sure predictions are finite
assert np.isfinite(final_pred).all()


print(
    "Prediction count:",
    len(final_pred)
)

print(
    "Prediction min:",
    final_pred.min()
)

print(
    "Prediction max:",
    final_pred.max()
)

print(
    "Prediction mean:",
    final_pred.mean()
)


# =========================================================
# 20. CREATE KAGGLE SUBMISSION
# =========================================================

submission = pd.DataFrame({

    "id": test["id"],

    "exam_score": final_pred

})


# =========================================================
# 21. VALIDATE SUBMISSION
# =========================================================

print("\n")
print("=" * 70)
print("SUBMISSION VALIDATION")
print("=" * 70)


print(
    "Submission shape:",
    submission.shape
)

print(
    "Expected rows:",
    len(test)
)

print("\nSubmission columns:")
print(
    submission.columns.tolist()
)

print("\nFirst 10 rows:")
print(
    submission.head(10)
)


assert len(submission) == len(test)

assert list(
    submission.columns
) == [
    "id",
    "exam_score"
]

assert submission["exam_score"].notna().all()

assert np.isfinite(
    submission["exam_score"]
).all()


# =========================================================
# 22. SAVE SUBMISSION
# =========================================================

OUTPUT_PATH = (
    "/content/submission_ensemble.csv"
)

submission.to_csv(
    OUTPUT_PATH,
    index=False
)


print("\n")
print("=" * 70)
print("SUCCESS!")
print("=" * 70)

print(
    "Submission saved at:"
)

print(
    OUTPUT_PATH
)

print("\nFile is ready for Kaggle.")


DATA INFORMATION
Train shape: (630000, 13)
Test shape : (270000, 12)

Train columns:
['id', 'age', 'gender', 'course', 'study_hours', 'class_attendance', 'internet_access', 'sleep_hours', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty', 'exam_score']

Test columns:
['id', 'age', 'gender', 'course', 'study_hours', 'class_attendance', 'internet_access', 'sleep_hours', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']

TARGET CHECK
NaN target values: 0
Infinite target values: 0
Removed invalid target rows: 0
Target validation: PASSED

X shape     : (630000, 11)
X_test shape: (270000, 11)

Categorical columns:
['gender', 'course', 'internet_access', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']

Numerical columns:
['age', 'study_hours', 'class_attendance', 'sleep_hours']

MISSING VALUE HANDLING

Missing values in X:
age                 0
gender              0
course              0
study_hours         0
class_attendance    